# 🎯 Reranking Techniques - The Secret to Top-Tier RAG

## What is Reranking?

**Reranking refines initial retrieval results using more sophisticated (but slower) models to improve relevance.**

### The Two-Stage Approach:

```
Stage 1: RETRIEVAL (Fast, cast wide net)
Query → Dense/Hybrid → Top 100 candidates
        ↓
Stage 2: RERANKING (Slow, precise)
100 candidates → Cross-Encoder → Top 5 best
```

![Reranking Pipeline](./images/reranking_pipeline.png)
*Figure 1: Multi-stage retrieval with reranking*

## Why Reranking Matters 🚀

### The Problem:
- **Bi-encoders** (BERT, E5): Fast but less accurate
  - Encode query and doc separately
  - No query-doc interaction
  - Can miss nuanced relevance

### The Solution:
- **Cross-encoders**: Slow but very accurate
  - Encode query+doc together
  - Full attention between them
  - Better relevance judgment

### Real-World Impact:
```
Retrieval alone:      NDCG@10 = 0.75
Retrieval + Rerank:   NDCG@10 = 0.85  (+13%! 🎯)
```

## Reranking Methods

### 1. **Cross-Encoder Reranking** ⭐ Most Popular
```python
score = cross_encoder(query, document)
# Full attention between query and document
```

### 2. **LLM-based Reranking**
```python
prompt = f"Rate relevance of doc to query: {query} | {doc}"
score = llm(prompt)
```

### 3. **Learned-to-Rank (LTR)**
```python
features = extract_features(query, doc)
score = ranking_model(features)
```

### 4. **Ensemble Reranking**
```python
score = α×cross_encoder + β×bm25 + γ×dense
```

## 1. Cross-Encoder Reranking

In [1]:
# Install dependencies
# !pip install sentence-transformers rank-bm25

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import numpy as np
import nltk

nltk.download('punkt', quiet=True)

# Sample documents
documents = [
    "Python is a high-level programming language widely used in data science and web development.",
    "Machine learning algorithms can learn patterns from data without explicit programming.",
    "Natural language processing helps computers understand and generate human language.",
    "Deep learning uses artificial neural networks with multiple layers to process information.",
    "RAG systems enhance language models by retrieving relevant context before generation.",
    "Vector databases store embeddings for efficient similarity search in high dimensions.",
    "Cross-encoders provide more accurate relevance scores but are slower than bi-encoders.",
    "Reranking improves retrieval quality by refining initial search results.",
    "The transformer architecture revolutionized natural language processing tasks.",
    "Fine-tuning adapts pre-trained models to specific downstream applications.",
]

print(f"📚 {len(documents)} documents loaded")

📚 10 documents loaded


In [2]:
# Stage 1: Initial retrieval with bi-encoder
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = bi_encoder.encode(documents)

query = "How to improve search results in RAG systems?"
query_embedding = bi_encoder.encode(query)

# Calculate similarities
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity([query_embedding], doc_embeddings)[0]

# Get top-5
top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

print(f"Query: '{query}'\n")
print("Stage 1 - Bi-Encoder Retrieval (Initial):")
print("="*80)

for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. [Score: {scores[idx]:.4f}] {documents[idx][:70]}...")

Query: 'How to improve search results in RAG systems?'

Stage 1 - Bi-Encoder Retrieval (Initial):
1. [Score: 0.5290] RAG systems enhance language models by retrieving relevant context bef...
2. [Score: 0.3996] Reranking improves retrieval quality by refining initial search result...
3. [Score: 0.3701] Cross-encoders provide more accurate relevance scores but are slower t...
4. [Score: 0.2243] Vector databases store embeddings for efficient similarity search in h...
5. [Score: 0.1326] Natural language processing helps computers understand and generate hu...


In [3]:
# Stage 2: Rerank with cross-encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Get top-k documents from stage 1
candidates = [documents[idx] for idx in top_indices]

# Create query-document pairs
pairs = [[query, doc] for doc in candidates]

# Get cross-encoder scores
cross_scores = cross_encoder.predict(pairs)

# Rerank based on cross-encoder scores
reranked_indices = np.argsort(cross_scores)[::-1]

print("\nStage 2 - Cross-Encoder Reranking (Refined):")
print("="*80)

for rank, idx in enumerate(reranked_indices, 1):
    original_idx = top_indices[idx]
    print(f"{rank}. [Score: {cross_scores[idx]:.4f}] {documents[original_idx][:70]}...")

print("\n💡 Notice: Order may change after reranking for better relevance!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Stage 2 - Cross-Encoder Reranking (Refined):
1. [Score: 1.7683] RAG systems enhance language models by retrieving relevant context bef...
2. [Score: -3.0114] Reranking improves retrieval quality by refining initial search result...
3. [Score: -10.7785] Vector databases store embeddings for efficient similarity search in h...
4. [Score: -11.3089] Cross-encoders provide more accurate relevance scores but are slower t...
5. [Score: -11.4215] Natural language processing helps computers understand and generate hu...

💡 Notice: Order may change after reranking for better relevance!


## 2. Complete Reranking Pipeline

In [4]:
class RerankingRetriever:
    def __init__(self,
                 retrieval_model='all-MiniLM-L6-v2',
                 reranking_model='cross-encoder/ms-marco-MiniLM-L-6-v2',
                 retrieval_top_k=20,
                 final_top_k=5):
        """
        Two-stage retrieval with reranking
        
        Parameters:
        - retrieval_top_k: How many to retrieve initially (cast wide net)
        - final_top_k: How many to return after reranking
        """
        self.bi_encoder = SentenceTransformer(retrieval_model)
        self.cross_encoder = CrossEncoder(reranking_model)
        self.retrieval_top_k = retrieval_top_k
        self.final_top_k = final_top_k
        
        self.documents = []
        self.doc_embeddings = None
        
    def index(self, documents):
        """Index documents"""
        self.documents = documents
        self.doc_embeddings = self.bi_encoder.encode(
            documents,
            show_progress_bar=True,
            batch_size=32
        )
        print(f"✅ Indexed {len(documents)} documents")
    
    def search(self, query, return_scores=True):
        """Two-stage search with reranking"""
        # Stage 1: Initial retrieval
        query_embedding = self.bi_encoder.encode(query)
        initial_scores = cosine_similarity(
            [query_embedding], 
            self.doc_embeddings
        )[0]
        
        # Get top-k candidates
        top_indices = np.argsort(initial_scores)[::-1][:self.retrieval_top_k]
        candidates = [self.documents[idx] for idx in top_indices]
        
        # Stage 2: Rerank with cross-encoder
        pairs = [[query, doc] for doc in candidates]
        cross_scores = self.cross_encoder.predict(pairs)
        
        # Get final top-k
        reranked_indices = np.argsort(cross_scores)[::-1][:self.final_top_k]
        
        # Build results
        results = []
        for rank, idx in enumerate(reranked_indices):
            original_idx = top_indices[idx]
            results.append({
                'rank': rank + 1,
                'rerank_score': float(cross_scores[idx]),
                'retrieval_score': float(initial_scores[original_idx]),
                'document': self.documents[original_idx],
                'index': int(original_idx)
            })
        
        return results

# Test the reranking retriever
reranker = RerankingRetriever(
    retrieval_top_k=10,  # Retrieve 10 candidates
    final_top_k=3        # Return top 3 after reranking
)

reranker.index(documents)

query = "What techniques improve RAG retrieval quality?"
results = reranker.search(query)

print(f"\nQuery: '{query}'\n")
print("Reranked Results:")
print("="*80)

for r in results:
    print(f"\n{r['rank']}. [Rerank: {r['rerank_score']:.4f}, Initial: {r['retrieval_score']:.4f}]")
    print(f"   {r['document']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 10 documents

Query: 'What techniques improve RAG retrieval quality?'

Reranked Results:

1. [Rerank: 3.1462, Initial: 0.2145]
   Reranking improves retrieval quality by refining initial search results.

2. [Rerank: 1.4746, Initial: 0.3935]
   RAG systems enhance language models by retrieving relevant context before generation.

3. [Rerank: -11.2992, Initial: 0.2326]
   Cross-encoders provide more accurate relevance scores but are slower than bi-encoders.


## 3. Performance Comparison

In [5]:
import time

# Test queries
test_queries = [
    "How does machine learning work?",
    "What is the best way to store embeddings?",
    "Explain transformers in NLP",
]

print("Performance Comparison: Bi-Encoder vs Reranking\n")
print("="*90)

for query in test_queries:
    # Bi-encoder only
    start = time.time()
    query_emb = bi_encoder.encode(query)
    scores = cosine_similarity([query_emb], doc_embeddings)[0]
    top_idx_bi = np.argmax(scores)
    time_bi = time.time() - start
    
    # With reranking
    start = time.time()
    results_rerank = reranker.search(query, return_scores=True)
    time_rerank = time.time() - start
    
    print(f"\nQuery: '{query}'")
    print(f"  Bi-encoder only: {time_bi*1000:.1f}ms → {documents[top_idx_bi][:50]}...")
    print(f"  With reranking:  {time_rerank*1000:.1f}ms → {results_rerank[0]['document'][:50]}...")
    
    if top_idx_bi != results_rerank[0]['index']:
        print(f"  ⚡ Reranking changed the top result!")

print("\n💡 Reranking is slower but often finds better results!")

Performance Comparison: Bi-Encoder vs Reranking


Query: 'How does machine learning work?'
  Bi-encoder only: 597.0ms → Machine learning algorithms can learn patterns fro...
  With reranking:  335.2ms → Machine learning algorithms can learn patterns fro...

Query: 'What is the best way to store embeddings?'
  Bi-encoder only: 397.9ms → Vector databases store embeddings for efficient si...
  With reranking:  853.7ms → Vector databases store embeddings for efficient si...

Query: 'Explain transformers in NLP'
  Bi-encoder only: 177.2ms → The transformer architecture revolutionized natura...
  With reranking:  285.8ms → The transformer architecture revolutionized natura...

💡 Reranking is slower but often finds better results!


## 4. Multi-Stage Retrieval Strategy

In [6]:
class MultiStageRetriever:
    def __init__(self):
        # Stage 1: BM25 (sparse)
        self.bm25 = None
        
        # Stage 2: Bi-encoder (dense)
        self.bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Stage 3: Cross-encoder (reranking)
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
        self.documents = []
        self.doc_embeddings = None
    
    def index(self, documents):
        self.documents = documents
        
        # BM25 index
        tokenized = [word_tokenize(doc.lower()) for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
        
        # Dense index
        self.doc_embeddings = self.bi_encoder.encode(documents)
        
        print(f"✅ Multi-stage index created for {len(documents)} docs")
    
    def search(self, query, stage1_k=50, stage2_k=20, final_k=5):
        """
        Three-stage retrieval:
        1. BM25: Get top 50 (fast recall)
        2. Dense: Rerank to top 20 (semantic precision)
        3. Cross-encoder: Final top 5 (highest accuracy)
        """
        # Stage 1: BM25 retrieval
        tokenized_query = word_tokenize(query.lower())
        bm25_scores = self.bm25.get_scores(tokenized_query)
        stage1_indices = np.argsort(bm25_scores)[::-1][:stage1_k]
        
        # Stage 2: Dense reranking
        stage1_docs = [self.documents[idx] for idx in stage1_indices]
        stage1_embeddings = self.doc_embeddings[stage1_indices]
        
        query_emb = self.bi_encoder.encode(query)
        dense_scores = cosine_similarity([query_emb], stage1_embeddings)[0]
        stage2_indices = np.argsort(dense_scores)[::-1][:stage2_k]
        
        # Stage 3: Cross-encoder final ranking
        stage2_docs = [stage1_docs[idx] for idx in stage2_indices]
        pairs = [[query, doc] for doc in stage2_docs]
        cross_scores = self.cross_encoder.predict(pairs)
        
        final_indices = np.argsort(cross_scores)[::-1][:final_k]
        
        # Build results
        results = []
        for rank, idx in enumerate(final_indices):
            stage2_idx = stage2_indices[idx]
            original_idx = stage1_indices[stage2_idx]
            
            results.append({
                'rank': rank + 1,
                'cross_score': float(cross_scores[idx]),
                'dense_score': float(dense_scores[stage2_idx]),
                'bm25_score': float(bm25_scores[original_idx]),
                'document': self.documents[original_idx]
            })
        
        return results

# Test multi-stage retrieval
multi_stage = MultiStageRetriever()
multi_stage.index(documents)

query = "improving search quality in AI systems"
results = multi_stage.search(query, stage1_k=8, stage2_k=5, final_k=3)

print(f"Query: '{query}'\n")
print("Multi-Stage Results:")
print("="*90)

for r in results:
    print(f"\n{r['rank']}. Scores [Cross: {r['cross_score']:.3f}, Dense: {r['dense_score']:.3f}, BM25: {r['bm25_score']:.3f}]")
    print(f"   {r['document']}")

print("\n💡 Each stage refines results for optimal quality!")

✅ Multi-stage index created for 10 docs
Query: 'improving search quality in AI systems'

Multi-Stage Results:

1. Scores [Cross: -0.538, Dense: 0.482, BM25: 3.261]
   Reranking improves retrieval quality by refining initial search results.

2. Scores [Cross: -10.066, Dense: 0.279, BM25: 2.401]
   Vector databases store embeddings for efficient similarity search in high dimensions.

3. Scores [Cross: -11.289, Dense: 0.339, BM25: 0.000]
   Cross-encoders provide more accurate relevance scores but are slower than bi-encoders.

💡 Each stage refines results for optimal quality!


## 5. Different Cross-Encoder Models

In [7]:
# Compare different cross-encoder models
cross_encoder_models = [
    'cross-encoder/ms-marco-TinyBERT-L-2-v2',      # Fastest
    'cross-encoder/ms-marco-MiniLM-L-6-v2',        # Balanced
    'cross-encoder/ms-marco-MiniLM-L-12-v2',       # Better quality
]

query = "machine learning and deep learning"
test_doc = documents[1]  # ML-related doc

print(f"Query: '{query}'")
print(f"Document: '{test_doc}'\n")
print(f"{'Model':<50} {'Score':<10} {'Time'}")
print("="*75)

for model_name in cross_encoder_models:
    ce = CrossEncoder(model_name)
    
    start = time.time()
    score = ce.predict([[query, test_doc]])[0]
    elapsed = time.time() - start
    
    model_short = model_name.split('/')[-1]
    print(f"{model_short:<50} {score:<10.4f} {elapsed*1000:.1f}ms")

print("\n💡 TinyBERT is 3-5x faster, MiniLM-L-12 is more accurate")

Query: 'machine learning and deep learning'
Document: 'Machine learning algorithms can learn patterns from data without explicit programming.'

Model                                              Score      Time


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

ms-marco-MiniLM-L-12-v2                            1.4790     76.9ms

💡 TinyBERT is 3-5x faster, MiniLM-L-12 is more accurate


## 6. Production Reranking System

In [8]:
from typing import List, Dict, Optional
import pickle

class ProductionReranker:
    def __init__(self,
                 retrieval_model='all-MiniLM-L-6-v2',
                 rerank_model='cross-encoder/ms-marco-MiniLM-L-6-v2',
                 retrieval_top_k=50,
                 rerank_top_k=20,
                 final_top_k=5,
                 rerank_batch_size=16):
        
        self.bi_encoder = SentenceTransformer(retrieval_model)
        self.cross_encoder = CrossEncoder(rerank_model)
        
        self.retrieval_top_k = retrieval_top_k
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k
        self.rerank_batch_size = rerank_batch_size
        
        self.documents = []
        self.metadata = []
        self.doc_embeddings = None
    
    def index(self, documents: List[str], metadata: Optional[List[Dict]] = None):
        self.documents = documents
        self.metadata = metadata if metadata else [{} for _ in documents]
        
        self.doc_embeddings = self.bi_encoder.encode(
            documents,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        
        print(f"✅ Indexed {len(documents)} documents for reranking")
    
    def search(self, query: str, filters: Optional[Dict] = None):
        # Stage 1: Dense retrieval
        query_emb = self.bi_encoder.encode(query, normalize_embeddings=True)
        scores = np.dot(self.doc_embeddings, query_emb)
        
        # Apply filters if provided
        if filters:
            valid_indices = [
                i for i, meta in enumerate(self.metadata)
                if all(meta.get(k) == v for k, v in filters.items())
            ]
            if valid_indices:
                scores = scores[valid_indices]
                filtered_docs = [self.documents[i] for i in valid_indices]
                filtered_meta = [self.metadata[i] for i in valid_indices]
            else:
                return []
        else:
            valid_indices = list(range(len(self.documents)))
            filtered_docs = self.documents
            filtered_meta = self.metadata
        
        # Get top candidates
        top_k = min(self.retrieval_top_k, len(scores))
        top_indices = np.argsort(scores)[::-1][:top_k]
        candidates = [filtered_docs[idx] for idx in top_indices]
        
        # Stage 2: Rerank top candidates
        rerank_k = min(self.rerank_top_k, len(candidates))
        if rerank_k > 0:
            pairs = [[query, doc] for doc in candidates[:rerank_k]]
            
            # Batch prediction for efficiency
            cross_scores = self.cross_encoder.predict(
                pairs,
                batch_size=self.rerank_batch_size,
                show_progress_bar=False
            )
            
            # Get final top-k
            final_k = min(self.final_top_k, len(cross_scores))
            reranked_indices = np.argsort(cross_scores)[::-1][:final_k]
        else:
            reranked_indices = []
        
        # Build results
        results = []
        for rank, idx in enumerate(reranked_indices):
            candidate_idx = top_indices[idx]
            original_idx = valid_indices[candidate_idx]
            
            results.append({
                'rank': rank + 1,
                'rerank_score': float(cross_scores[idx]),
                'retrieval_score': float(scores[candidate_idx]),
                'document': self.documents[original_idx],
                'metadata': self.metadata[original_idx],
                'index': original_idx
            })
        
        return results
    
    def save(self, path: str):
        np.savez(
            f"{path}.npz",
            embeddings=self.doc_embeddings,
            documents=self.documents,
            metadata=self.metadata
        )
        print(f"✅ Saved to {path}.npz")
    
    def load(self, path: str):
        data = np.load(f"{path}.npz", allow_pickle=True)
        self.doc_embeddings = data['embeddings']
        self.documents = data['documents'].tolist()
        self.metadata = data['metadata'].tolist()
        print(f"✅ Loaded from {path}.npz")

# Test production reranker
prod_reranker = ProductionReranker(
    retrieval_top_k=8,
    rerank_top_k=5,
    final_top_k=3
)

# Add metadata
metadata = [
    {"category": "programming", "difficulty": "easy"},
    {"category": "AI", "difficulty": "medium"},
    {"category": "AI", "difficulty": "medium"},
    {"category": "AI", "difficulty": "hard"},
    {"category": "RAG", "difficulty": "medium"},
    {"category": "databases", "difficulty": "medium"},
    {"category": "RAG", "difficulty": "hard"},
    {"category": "RAG", "difficulty": "medium"},
    {"category": "AI", "difficulty": "medium"},
    {"category": "AI", "difficulty": "medium"},
]

prod_reranker.index(documents, metadata=metadata)

# Search with filter
results = prod_reranker.search(
    "improving retrieval in RAG",
    filters={"category": "RAG"}
)

print("\nFiltered Reranked Results:")
for r in results:
    print(f"\n{r['rank']}. [Rerank: {r['rerank_score']:.3f}]")
    print(f"   {r['document']}")
    print(f"   {r['metadata']}")

No sentence-transformers model found with name sentence-transformers/all-MiniLM-L-6-v2. Creating a new one with mean pooling.


OSError: sentence-transformers/all-MiniLM-L-6-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## Key Takeaways 🎯

### ✅ Reranking Essentials:

1. **Two-Stage Pipeline**: Fast retrieval → Slow reranking
2. **Cross-Encoders**: Full attention for better relevance
3. **Trade-off**: Speed vs Quality (worth it!)
4. **Typical Gains**: 10-20% improvement in relevance
5. **Production Standard**: Used by all major search engines

### 🔑 When to Use Reranking:

**Always use when:**
- ✅ Quality matters more than speed
- ✅ Building production RAG
- ✅ Have >100 candidate documents
- ✅ Complex queries need nuanced matching

**Skip if:**
- ❌ Real-time requirements (<50ms)
- ❌ Very few candidates (<10)
- ❌ Simple exact matching tasks

### 📊 Architecture Patterns:

```python
# Pattern 1: Two-Stage (Most Common)
Stage 1: Dense retrieval → Top 50-100
Stage 2: Cross-encoder → Top 5-10

# Pattern 2: Three-Stage (Best Quality)
Stage 1: BM25 → Top 100 (recall)
Stage 2: Dense → Top 20 (semantic)
Stage 3: Cross-encoder → Top 5 (precision)

# Pattern 3: Hybrid + Rerank
Stage 1: Hybrid (BM25 + Dense) → Top 50
Stage 2: Cross-encoder → Top 5
```

### 🚀 Model Selection:

| Model | Speed | Quality | Best For |
|-------|-------|---------|----------|
| **TinyBERT-L-2** | ⚡⚡⚡ | ⭐⭐ | High throughput |
| **MiniLM-L-6** | ⚡⚡ | ⭐⭐⭐ | **Recommended** |
| **MiniLM-L-12** | ⚡ | ⭐⭐⭐⭐ | Max quality |

**Recommendation: MiniLM-L-6 for production (best balance)**

### 💡 Performance Optimization:

```python
# 1. Batch predictions
cross_encoder.predict(pairs, batch_size=16)  # Much faster!

# 2. Limit candidates
retrieval_top_k = 50  # Don't rerank everything
rerank_top_k = 20     # Only top 20 need reranking

# 3. Use GPU if available
cross_encoder = CrossEncoder(model, device='cuda')

# 4. Cache for repeated queries
from functools import lru_cache
@lru_cache(maxsize=1000)
def cached_rerank(query, doc):
    return cross_encoder.predict([[query, doc]])
```

### 📈 Expected Improvements:

```
Metric improvements with reranking:

NDCG@10:     +10-15%
MRR:         +15-20%
Precision@5: +20-25%

Speed impact:
- Retrieval: 50ms
- Reranking: +200-500ms
- Total: 250-550ms

Worth it for most applications! ✅
```

### 🎯 Production Checklist:

- [ ] Choose cross-encoder model (MiniLM-L-6 recommended)
- [ ] Set retrieval_top_k (50-100 typical)
- [ ] Set rerank_top_k (10-20 typical)
- [ ] Set final_top_k (5-10 typical)
- [ ] Enable batch prediction
- [ ] Use GPU if available
- [ ] Add query caching
- [ ] Monitor latency
- [ ] A/B test impact

### 🔬 Advanced Techniques:

```python
# 1. Score Fusion
final_score = α×cross_score + β×dense_score + γ×bm25_score

# 2. Diversity Reranking
results = rerank_with_diversity(candidates, lambda_diversity=0.5)

# 3. Contextual Reranking  
score = cross_encoder(query + conversation_history, doc)

# 4. Multi-vector Reranking
score = cross_encoder(query, doc_chunks).max()
```

### 🏆 Best Practice Pipeline:

```python
class BestPracticeRAG:
    def retrieve(self, query):
        # Stage 1: Hybrid retrieval (50-100 candidates)
        candidates = self.hybrid_retriever.search(query, top_k=100)
        
        # Stage 2: Cross-encoder reranking (top 20)
        reranked = self.cross_encoder.rerank(query, candidates, top_k=20)
        
        # Stage 3: Diversity filtering (final 5)
        diverse = self.diversify(reranked, top_k=5)
        
        return diverse
```

## Next Steps 🔜

You've mastered retrieval! Now move to:
- **`3_search_techniques/`** - Advanced search algorithms
- **`4_generation/`** - Integrate with LLMs
- **`6_evaluation/`** - Measure retrieval quality

Ready to build world-class RAG systems! 🚀